# Deep sequence models vs gradient boosting

Do models built for temporal data beat our gradient-boosting champion? Three standard architectures for C-MAPSS are trained on **windows of the last 30 cycles of per-condition-normalised sensors** (past only, checked by a causality test):

| Model | Idea |
|---|---|
| **1D-CNN** | stacked 1D convolutions over time (the classic C-MAPSS conv net) |
| **GRU** | two-layer recurrent network, read out at the latest cycle |
| **TCN** | causal dilated convolutions with residual blocks (receptive field 61 cycles) |

**Fair comparison**
- Same cleaning, per-condition normalisation, RUL cap (125) and engine-level splits as the pipeline. The nets get the raw normalised sensors plus `cycle`; the trees keep their engineered features. That's the point of the comparison.
- **The same 5-fold GroupKFold by engine** as the ablation study, so every fold is a paired comparison. Nets early-stop on 15% of each fold's training engines; HistGB and XGBoost are refit in the same folds.
- **Official test set:** every model is trained on the same engines as the production model (v2). The nets early-stop on the validation engines, which nothing trains on. One prediction per test engine at its last cycle.
- **3 random starts per net (seeds 42, 1, 2).** A net's score depends on its initial weights and batch order, so one seed could win or lose by luck. Each net is trained 3 times in every fold and on the test fit. The net as deployed is the average of its 3 seeds (a seed ensemble), and the seed-spread table shows how far single seeds scatter around it.
- **Blends:** HistGB + each net, all three nets, and HistGB + all three. A different model family may help where the HistGB + XGBoost blend didn't.
- **Verdict** (same spirit as the promotion gate): a net or blend replaces boosting only if its capped CV RMSE is lower by more than the 0.5-cycle margin in at least 4 of 5 folds. Otherwise boosting stays: it's faster, explainable and lighter.

**Run on Kaggle**: attach the dataset, *Accelerator → GPU T4* (one GPU is used), then *Save Version → Save & Run All*. Expect roughly 2.5–3 hours for all four subsets (3 nets × 3 seeds × 6 fits each); to split it across sessions, shorten `SUBSETS`. Output: `/kaggle/working/deep_outputs.zip`. Generated from `scripts/build_notebook.py`; edit it there.

In [ ]:
import json

# pipeline code cells + deep-model code, generated from scripts/build_notebook.py
DEEP_CELLS = json.loads(r'''[
"import json, shutil, subprocess, time, warnings\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport joblib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, IsolationForest\nfrom sklearn.linear_model import Ridge\nfrom sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nwarnings.filterwarnings(\"ignore\", category=UserWarning)\nwarnings.filterwarnings(\"ignore\", category=FutureWarning)\ntry:\n    import xgboost as xgb\nexcept ImportError:\n    xgb = None\n\nSUBSET = \"FD001\"  # FD001 / FD002 / FD003 / FD004 (run_all_subsets.ipynb rewrites this line)\nSEED = 42\nRUN_TAG = \"\"  # run_all_subsets sets e.g. \"_seed1\" for extra seeds, so their outputs don't overwrite seed 42's\n\nSUBSET_CONFIG = {\n    \"FD001\": {\"condition_norm\": False, \"fault_modes\": 1},\n    \"FD002\": {\"condition_norm\": True, \"fault_modes\": 1},\n    \"FD003\": {\"condition_norm\": False, \"fault_modes\": 2},\n    \"FD004\": {\"condition_norm\": True, \"fault_modes\": 2},\n}\n\nCFG = {\n    \"subset\": SUBSET, **SUBSET_CONFIG[SUBSET],\n    # labels\n    \"rul_cap\": 125,             # piecewise-linear target: flat while the engine is healthy\n    \"fail_window\": 30,          # \"failing\" class for the early-warning classifier: RUL <= 30\n    # cleaning / features\n    \"min_unique\": 2,            # drop sensors with <= 2 distinct values within a condition\n    \"clip_quantiles\": (0.001, 0.999),\n    # data-quality gate (all fitted on training engines)\n    \"op_margin_frac\": 0.5,        # single-condition subsets: settings beyond the training range by this much = unseen\n    \"sensor_range_margin\": 0.5,   # reading beyond the training min/max by 50% of the range = out of range\n    \"sensor_range_persist\": 2,    # ... for at least this many consecutive cycles\n    \"sensor_stuck_cycles\": 15,    # identical reading this long on a sensor that never flatlines in training = stuck\n    \"roll_window\": 10, \"slope_window\": 20, \"ewm_span\": 10, \"baseline_cycles\": 10,\n    # splits (by engine)\n    \"val_frac\": 0.20, \"new_frac\": 0.30,\n    \"cv_folds\": 5,\n    # 'test-like' evaluation rows: mirror how the official test engines are cut.\n    # Test RUL reaches ~195 in FD002/FD004, so the window covers RUL up to 200.\n    \"eval_min_cycles\": 31, \"eval_max_rul\": 200,\n    # anomaly detector: trained on early-life cycles (no failure labels needed; a real fleet would use the\n    # first cycles after commissioning or maintenance). Starts after the rolling window is full.\n    \"anomaly_train_cycles\": (11, 40), \"anomaly_quantile\": 0.99, \"anomaly_persist\": 3,\n    \"anomaly_healthy_rul\": 125,  # evaluation only: false-alarm rate is measured on RUL > 125 (a proxy)\n    # RUL interval: conformal calibration of the 10% / 90% quantile models\n    \"interval_coverage\": 0.80, \"review_width_quantile\": 0.90,\n    # retraining / promotion\n    \"retrain_min_new_frac\": 0.20, \"psi_alert\": 0.20, \"promote_max_nasa_ratio\": 1.10,\n    \"promote_min_prob\": 0.80,     # how sure the bootstrap must be (80% of resamples of the validation engines)\n    # retraining here is always \"same design, more data\", so v2 is promoted unless it is clearly WORSE\n    # (non-inferiority); \"superiority\" (v2 must be clearly better) is the stricter test for design changes\n    \"promotion_test\": \"non_inferiority\",\n    \"noninferiority_margin\": 0.5,  # cycles of capped RMSE: \"clearly worse\" means worse by more than this (< fold spread)\n    # after promotion, retrain the winning design on every training engine and deploy that\n    \"full_refit\": False,\n}\n\nON_KAGGLE = Path(\"/kaggle/input\").exists()\nWORK = Path(\"/kaggle/working\") if ON_KAGGLE else Path.cwd() / \"outputs\"\nOUT = WORK / \"artifacts\" / f\"{SUBSET}{RUN_TAG}\"\n(OUT / \"plots\").mkdir(parents=True, exist_ok=True)\n(OUT / \"models\").mkdir(parents=True, exist_ok=True)\nnp.random.seed(SEED)\n\n\ndef gpu_available():\n    try:\n        r = subprocess.run([\"nvidia-smi\", \"-L\"], capture_output=True, text=True)\n        return r.returncode == 0 and \"GPU\" in r.stdout\n    except FileNotFoundError:\n        return False\n\n\ndef now_iso():\n    return datetime.now(timezone.utc).isoformat(timespec=\"seconds\")\n\n\ndef json_default(o):\n    if isinstance(o, (np.integer,)):\n        return int(o)\n    if isinstance(o, (np.floating,)):\n        return float(o)\n    if isinstance(o, (np.bool_,)):\n        return bool(o)\n    if isinstance(o, (Path,)):\n        return str(o)\n    raise TypeError(type(o))\n\n\ndef save_json(obj, path):\n    Path(path).write_text(json.dumps(obj, indent=2, default=json_default))\n\n\nGPU = gpu_available()\nprint(f\"subset={SUBSET}  gpu={GPU}  xgboost={xgb.__version__ if xgb else 'not installed'}  outputs={OUT}\")",
"COLS = [\"unit\", \"cycle\", \"op1\", \"op2\", \"op3\"] + [f\"s{i}\" for i in range(1, 22)]\nOPS, SENSORS = COLS[2:5], COLS[5:]\n\n\ndef find_data_dir(subset):\n    roots = [Path(\"/kaggle/input\"), Path.cwd() / \"data\" / \"csv\", Path.cwd().parent / \"data\" / \"csv\",\n             Path.cwd() / \"DataSet\", Path.cwd().parent / \"DataSet\"]\n    for root in roots:\n        if root.exists():\n            for ext in (\"csv\", \"txt\"):\n                hits = sorted(root.rglob(f\"train_{subset}.{ext}\"))\n                if hits:\n                    return hits[0].parent, ext\n    raise FileNotFoundError(f\"train_{subset}.csv / .txt not found under {[str(r) for r in roots]}\")\n\n\ndef read_table(path, ext):\n    if ext == \"csv\":\n        return pd.read_csv(path)\n    df = pd.read_csv(path, sep=r\"\\s+\", header=None)\n    return df.iloc[:, :len(COLS)].set_axis(COLS, axis=1)\n\n\ndef validate_raw(df, name):\n    missing = [c for c in COLS if c not in df.columns]\n    if missing:\n        raise ValueError(f\"{name}: missing columns {missing}\")\n    num = df[COLS].apply(pd.to_numeric, errors=\"coerce\")\n    report = {\n        \"rows\": len(df),\n        \"engines\": int(num.unit.nunique()),\n        \"missing_cells\": int(df[COLS].isna().sum().sum()),\n        \"non_numeric_cells\": int(num.isna().sum().sum() - df[COLS].isna().sum().sum()),\n        \"duplicate_unit_cycle\": int(num.duplicated(subset=[\"unit\", \"cycle\"]).sum()),\n        \"cycle_gaps\": int(num.sort_values([\"unit\", \"cycle\"]).groupby(\"unit\").cycle.diff().dropna().ne(1).sum()),\n    }\n    if report[\"rows\"] == 0:\n        raise ValueError(f\"{name}: empty file\")\n    return report\n\n\ndef load_subset(subset):\n    d, ext = find_data_dir(subset)\n    train = read_table(d / f\"train_{subset}.{ext}\", ext)\n    test = read_table(d / f\"test_{subset}.{ext}\", ext)\n    if ext == \"csv\":\n        r = pd.read_csv(d / f\"RUL_{subset}.csv\")\n        final = (r.sort_values(\"unit\") if \"unit\" in r else r)[\"rul\"].to_numpy()\n    else:\n        final = pd.read_csv(d / f\"RUL_{subset}.txt\", sep=r\"\\s+\", header=None)[0].to_numpy()\n    reports = {\"train\": validate_raw(train, \"train\"), \"test\": validate_raw(test, \"test\")}\n    test_units = np.sort(pd.to_numeric(test.unit).unique())\n    if len(final) != len(test_units):\n        raise ValueError(f\"RUL file has {len(final)} values but test has {len(test_units)} engines\")\n    rul_final = pd.DataFrame({\"unit\": test_units.astype(int), \"rul_final\": final.astype(int)})\n    return train[COLS], test[COLS], rul_final, reports, d\n\n\nraw_train, raw_test, rul_final, ingest_report, DATA_DIR = load_subset(SUBSET)\nprint(\"data dir:\", DATA_DIR)\ndisplay(pd.DataFrame(ingest_report).T)",
"def basic_clean(df):\n    df = df[COLS].apply(pd.to_numeric, errors=\"coerce\").dropna(subset=[\"unit\", \"cycle\"])\n    df[[\"unit\", \"cycle\"]] = df[[\"unit\", \"cycle\"]].astype(int)\n    df = df.drop_duplicates(subset=[\"unit\", \"cycle\"], keep=\"last\").sort_values([\"unit\", \"cycle\"]).reset_index(drop=True)\n    vals = OPS + SENSORS\n    df[\"imputed\"] = df[vals].isna().any(axis=1)\n    # past-only: forward fill within each engine; leading gaps become the training mean after normalisation\n    df[vals] = df.groupby(\"unit\")[vals].ffill()\n    return df\n\n\ndef add_labels(df, cfg, rul_final=None):\n    df = df.copy()\n    last = df.groupby(\"unit\").cycle.transform(\"max\")\n    end = 0 if rul_final is None else df.unit.map(rul_final.set_index(\"unit\").rul_final)\n    df[\"rul\"] = (end + last - df.cycle).astype(int)\n    df[\"rul_capped\"] = df.rul.clip(upper=cfg[\"rul_cap\"])\n    df[\"fail\"] = (df.rul <= cfg[\"fail_window\"]).astype(int)\n    return df\n\n\ntrain_df = add_labels(basic_clean(raw_train), CFG)\ntest_df = add_labels(basic_clean(raw_test), CFG, rul_final)\n\ncheck = test_df.groupby(\"unit\").rul.min().reset_index().merge(rul_final, on=\"unit\")\nassert (check.rul == check.rul_final).all(), \"test labels do not match the RUL file\"\nprint(f\"train: {train_df.unit.nunique()} engines, {len(train_df):,} rows | test: {test_df.unit.nunique()} engines, {len(test_df):,} rows\")\nprint(f\"imputed rows: train {int(train_df.imputed.sum())}, test {int(test_df.imputed.sum())}\")\nprint(f\"class balance (fail = RUL <= {CFG['fail_window']}): {train_df.fail.mean():.1%} positive\")",
"META = [\"unit\", \"cycle\", \"condition\", \"imputed\", \"unseen_condition\", \"sensor_fault\", \"sensor_fault_detail\",\n        \"rul\", \"rul_capped\", \"fail\"]\n\n\ndef condition_key(df):\n    return (df[OPS].round(0) + 0.0).astype(str).agg(\"/\".join, axis=1)\n\n\ndef assign_condition(df, prep):\n    # operating condition per row; settings never seen in training fall back to the nearest known condition and\n    # are flagged, instead of silently using global statistics\n    if not prep[\"condition_norm\"]:\n        lo = pd.Series({o: prep[\"op_range\"][o][0] for o in OPS})\n        hi = pd.Series({o: prep[\"op_range\"][o][1] for o in OPS})\n        margin = (hi - lo) * prep[\"quality\"][\"op_margin_frac\"] + 0.01 * np.maximum(1.0, np.maximum(lo.abs(), hi.abs()))\n        unseen = ((df[OPS] < lo - margin) | (df[OPS] > hi + margin)).any(axis=1)\n        return pd.Series(\"all\", index=df.index), unseen\n    key = condition_key(df)\n    unseen = ~key.isin(list(prep[\"stats\"]))\n    if unseen.any():\n        names = list(prep[\"centers\"])\n        centers = np.array([prep[\"centers\"][n] for n in names], dtype=float)\n        scale = np.where(centers.std(axis=0) > 0, centers.std(axis=0), 1.0)\n        x = df.loc[unseen, OPS].to_numpy(float)\n        dist = np.nansum(((x[:, None, :] - centers[None]) / scale) ** 2, axis=2)\n        nearest = np.array(names, dtype=object)[dist.argmin(axis=1)]\n        nearest[np.isnan(x).all(axis=1)] = prep[\"default_condition\"]\n        key = key.copy()\n        key.loc[unseen] = nearest\n    return key, unseen\n\n\ndef _flat(frame, units, window):\n    # identical for the whole window: max - min is exactly 0 (a rolling std leaves rounding residue up to ~1e-5)\n    return (_rolling_by_unit(frame, units, window, \"max\") - _rolling_by_unit(frame, units, window, \"min\")) == 0\n\n\ndef _rolling_by_unit(frame, units, window, how):\n    # per-engine trailing window statistic, aligned back to the frame's row order\n    r = frame.groupby(units.to_numpy()).rolling(window, min_periods=window)\n    return getattr(r, how)().reset_index(level=0, drop=True).reindex(frame.index)\n\n\ndef sensor_quality(df, prep):\n    # past-only sensor-health checks: dead (never reported), stuck (flat for K cycles on a sensor that never\n    # flatlines in training) and out of the physically seen range for >= persist cycles\n    q, sensors = prep[\"quality\"], prep[\"sensors\"]\n    raw = df[sensors].astype(float)\n    lo = pd.DataFrame(prep[\"sensor_range\"][\"lo\"]).T.loc[df[\"condition\"], sensors].to_numpy()\n    hi = pd.DataFrame(prep[\"sensor_range\"][\"hi\"]).T.loc[df[\"condition\"], sensors].to_numpy()\n    span, x = hi - lo, raw.to_numpy()\n    oor = (x < lo - q[\"range_margin\"] * span) | (x > hi + q[\"range_margin\"] * span)\n    # an unseen operating condition shifts every reading; that is flagged separately, not blamed on the sensors\n    oor &= ~df[\"unseen_condition\"].to_numpy()[:, None]\n    oor = pd.DataFrame(oor, index=df.index, columns=sensors).astype(float)\n    oor = _rolling_by_unit(oor, df[\"unit\"], q[\"range_persist\"], \"min\").fillna(0).astype(bool)\n    stuck = pd.DataFrame(False, index=df.index, columns=sensors)\n    if prep[\"stuck_sensors\"]:\n        stuck[prep[\"stuck_sensors\"]] = _flat(raw[prep[\"stuck_sensors\"]], df[\"unit\"], q[\"stuck_cycles\"])\n    dead = raw.isna()\n    fault = (oor | stuck | dead).any(axis=1)\n    detail = pd.Series(\"\", index=df.index)\n    for i in np.flatnonzero(fault.to_numpy()):\n        row = df.index[i]\n        parts = [f\"{s} dead\" for s in sensors if dead.at[row, s]] + \\\n                [f\"{s} stuck\" for s in sensors if stuck.at[row, s]] + \\\n                [f\"{s} out of range\" for s in sensors if oor.at[row, s]]\n        detail.at[row] = \"; \".join(parts)\n    return fault, detail\n\n\ndef _normalise(df, prep, sensors):\n    out = pd.DataFrame(index=df.index, columns=sensors, dtype=float)\n    for c, g in df.groupby(\"condition\"):\n        st_ = prep[\"stats\"].get(c, prep[\"global\"])\n        mu = pd.Series(st_[\"mean\"])[sensors]\n        sd = pd.Series(st_[\"std\"])[sensors].replace(0, np.nan)\n        out.loc[g.index] = ((g[sensors] - mu) / sd).to_numpy()\n    return out.fillna(0.0)\n\n\ndef fit_prep(df, cfg):\n    df = df.assign(condition=condition_key(df) if cfg[\"condition_norm\"] else \"all\")\n    stats = {c: {\"mean\": g[SENSORS].mean().to_dict(), \"std\": g[SENSORS].std().to_dict()}\n             for c, g in df.groupby(\"condition\")}\n    glob = {\"mean\": df[SENSORS].mean().to_dict(), \"std\": df[SENSORS].std().to_dict()}\n    nunique = df.groupby(\"condition\")[SENSORS].nunique().median()\n    sensors = [s for s in SENSORS if nunique[s] > cfg[\"min_unique\"]]\n    prep = {\"sensors\": sensors, \"stats\": stats, \"global\": glob, \"condition_norm\": cfg[\"condition_norm\"]}\n    z = _normalise(df, prep, sensors)\n    lo, hi = cfg[\"clip_quantiles\"]\n    prep[\"clip\"] = {s: (float(z[s].quantile(lo)), float(z[s].quantile(hi))) for s in sensors}\n    prep[\"dropped\"] = [s for s in SENSORS if s not in sensors]\n    # what \"normal\" looks like, for the unseen-condition check and the sensor-health gate\n    grouped = df.groupby(\"condition\")\n    prep[\"centers\"] = {c: [float(v) for v in g[OPS].mean()] for c, g in grouped}\n    prep[\"default_condition\"] = str(df[\"condition\"].value_counts().idxmax())\n    prep[\"op_range\"] = {o: [float(df[o].min()), float(df[o].max())] for o in OPS}\n    prep[\"sensor_range\"] = {\"lo\": grouped[sensors].min().T.to_dict(), \"hi\": grouped[sensors].max().T.to_dict()}\n    k = cfg[\"sensor_stuck_cycles\"]\n    flat = _flat(df[sensors].astype(float), df[\"unit\"], k)\n    prep[\"stuck_sensors\"] = [s for s in sensors if flat[s].mean() < 1e-4]  # never flat in training -> checkable\n    prep[\"quality\"] = {\"op_margin_frac\": cfg[\"op_margin_frac\"], \"range_margin\": cfg[\"sensor_range_margin\"],\n                       \"range_persist\": cfg[\"sensor_range_persist\"], \"stuck_cycles\": k}\n    return prep\n\n\ndef apply_prep(df, prep):\n    df = df.sort_values([\"unit\", \"cycle\"]).reset_index(drop=True)\n    condition, unseen = assign_condition(df, prep)\n    df = df.assign(condition=condition, unseen_condition=unseen.to_numpy())\n    fault, detail = sensor_quality(df, prep)\n    df = df.assign(sensor_fault=fault.to_numpy(), sensor_fault_detail=detail.to_numpy())\n    sensors = prep[\"sensors\"]\n    z = _normalise(df, prep, sensors)\n    for s in sensors:\n        z[s] = z[s].clip(*prep[\"clip\"][s])\n    return pd.concat([df[[c for c in META if c in df]], z], axis=1)\n\n\ndef _rolling_sum(a, w):\n    c = np.cumsum(a, axis=0)\n    out = c.copy()\n    out[w:] = c[w:] - c[:-w]\n    return out\n\n\ndef _rolling_slope(y, w):\n    # least-squares slope over the last w cycles (fewer at the start of an engine's life)\n    n_rows = len(y)\n    t = np.arange(n_rows, dtype=float)[:, None]\n    n = np.minimum(np.arange(1, n_rows + 1), w)[:, None].astype(float)\n    st, stt = _rolling_sum(t, w), _rolling_sum(t * t, w)\n    sy, sty = _rolling_sum(y, w), _rolling_sum(t * y, w)\n    den = n * stt - st ** 2\n    with np.errstate(divide=\"ignore\", invalid=\"ignore\"):\n        slope = (n * sty - st * sy) / den\n    return np.where(den > 1e-9, slope, 0.0)\n\n\ndef feature_cols(prep, cfg):\n    w, sw, span = cfg[\"roll_window\"], cfg[\"slope_window\"], cfg[\"ewm_span\"]\n    sufs = [\"\", f\"_mean{w}\", f\"_std{w}\", f\"_slope{sw}\", f\"_ewm{span}\", \"_drift\"]\n    return [\"cycle\"] + [f\"{s}{suf}\" for suf in sufs for s in prep[\"sensors\"]]\n\n\ndef anomaly_cols(prep, cfg):\n    return [f\"{s}_mean{cfg['roll_window']}\" for s in prep[\"sensors\"]]\n\n\ndef build_features(dz, sensors, cfg):\n    w, sw, span, base_n = cfg[\"roll_window\"], cfg[\"slope_window\"], cfg[\"ewm_span\"], cfg[\"baseline_cycles\"]\n    blocks = []\n    for _, g in dz.groupby(\"unit\", sort=True):\n        y = g[sensors].to_numpy(float)\n        frame = pd.DataFrame(y)\n        roll = frame.rolling(w, min_periods=1)\n        ewm = frame.ewm(span=span, adjust=False).mean().to_numpy()\n        base = np.cumsum(y, axis=0) / np.arange(1, len(y) + 1)[:, None]\n        if len(y) > base_n:\n            base[base_n:] = base[base_n - 1]\n        blocks.append(np.hstack([y, roll.mean().to_numpy(), roll.std().fillna(0.0).to_numpy(),\n                                 _rolling_slope(y, sw), ewm, ewm - base]))\n    names = feature_cols({\"sensors\": sensors}, cfg)[1:]\n    X = pd.DataFrame(np.vstack(blocks), columns=names, index=dz.index)\n    return pd.concat([dz[[c for c in META if c in dz]], X], axis=1)\n\n\ndef featurize(df, prep, cfg):\n    return build_features(apply_prep(df, prep), prep[\"sensors\"], cfg)\n\n\ndef check_causality(df, prep, cfg, n_units=8, seed=SEED):\n    rng = np.random.default_rng(seed)\n    full = featurize(df, prep, cfg).set_index([\"unit\", \"cycle\"])\n    feats = feature_cols(prep, cfg)[1:]\n    for u in rng.choice(df.unit.unique(), n_units, replace=False):\n        g = df[df.unit == u]\n        cut = int(rng.integers(1, len(g) + 1))\n        part = featurize(g.iloc[:cut], prep, cfg).iloc[-1]\n        ref = full.loc[(u, int(part.cycle)), feats].to_numpy(float)\n        assert np.allclose(part[feats].to_numpy(float), ref, atol=1e-8), f\"feature leak at engine {u}\"\n    return True\n\n\ndef check_cleaning_causality(raw, prep, cfg, n_units=8, seed=SEED):\n    # inject gaps (dense at the start of the series), then check cleaning + features stay past-only\n    rng = np.random.default_rng(seed)\n    feats = feature_cols(prep, cfg)[1:]\n    for u in rng.choice(raw.unit.unique(), n_units, replace=False):\n        g = raw[raw.unit == u].copy()\n        gaps = rng.random((len(g), len(SENSORS))) < 0.05\n        gaps[:3] |= rng.random((3, len(SENSORS))) < 0.5\n        g[SENSORS] = g[SENSORS].mask(gaps)\n        full = featurize(basic_clean(g), prep, cfg).set_index(\"cycle\")\n        cut = int(rng.integers(1, len(g) + 1))\n        part = featurize(basic_clean(g.iloc[:cut]), prep, cfg).iloc[-1]\n        ref = full.loc[int(part.cycle), feats].to_numpy(float)\n        assert np.allclose(part[feats].to_numpy(float), ref, atol=1e-8), f\"cleaning leak at engine {u}\"\n    return True\n\n\n_prep_demo = fit_prep(train_df, CFG)\n_feat_demo = featurize(train_df, _prep_demo, CFG)\nprint(\"sensors kept   :\", _prep_demo[\"sensors\"])\nprint(\"sensors dropped:\", _prep_demo[\"dropped\"])\nprint(\"conditions     :\", len(_prep_demo[\"stats\"]))\nprint(f\"features       : {len(feature_cols(_prep_demo, CFG))} per row, matrix {_feat_demo.shape}\")\nassert not _feat_demo[feature_cols(_prep_demo, CFG)].isna().any().any()\nprint(\"causality check:\", check_causality(train_df, _prep_demo, CFG))\nprint(\"cleaning causality check (injected gaps):\", check_cleaning_causality(raw_train, _prep_demo, CFG))",
"def split_units(df, cfg, seed=SEED):\n    units = np.array(sorted(df.unit.unique()))\n    np.random.default_rng(seed).shuffle(units)\n    n_val, n_new = round(len(units) * cfg[\"val_frac\"]), round(len(units) * cfg[\"new_frac\"])\n    return {\"val\": sorted(units[:n_val].tolist()),\n            \"new\": sorted(units[n_val:n_val + n_new].tolist()),\n            \"initial\": sorted(units[n_val + n_new:].tolist())}\n\n\nSPLITS = split_units(train_df, CFG)\nassert not (set(SPLITS[\"val\"]) & set(SPLITS[\"new\"]) | set(SPLITS[\"val\"]) & set(SPLITS[\"initial\"])\n            | set(SPLITS[\"new\"]) & set(SPLITS[\"initial\"])), \"engine leak between splits\"\nval_df = train_df[train_df.unit.isin(SPLITS[\"val\"])].reset_index(drop=True)\nprint({k: len(v) for k, v in SPLITS.items()}, \"engines\")",
"def make_rul_model(kind):\n    if kind == \"ridge\":\n        return make_pipeline(StandardScaler(), Ridge(alpha=1.0))\n    if kind == \"hgb\":\n        return HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20,\n                                             l2_regularization=1.0, early_stopping=False, random_state=SEED)\n    if kind == \"xgb\":\n        return xgb.XGBRegressor(n_estimators=600, learning_rate=0.03, max_depth=5, subsample=0.8,\n                                colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0, tree_method=\"hist\",\n                                device=\"cuda\" if GPU else \"cpu\", random_state=SEED, n_jobs=-1)\n    raise ValueError(kind)\n\n\ndef make_quantile(q):\n    return HistGradientBoostingRegressor(loss=\"quantile\", quantile=q, max_iter=300, learning_rate=0.05,\n                                         max_leaf_nodes=31, min_samples_leaf=20, early_stopping=False,\n                                         random_state=SEED)\n\n\ndef make_classifier():\n    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20,\n                                          class_weight=\"balanced\", early_stopping=False, random_state=SEED)\n\n\nCANDIDATES = [\"ridge\", \"hgb\"] + ([\"xgb\"] if xgb is not None else [])\n\n\ndef nasa_score(y_true, y_pred):\n    d = np.asarray(y_pred, float) - np.asarray(y_true, float)\n    return np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)\n\n\ndef rul_metrics(y_true, y_pred, cap):\n    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)\n    d = y_pred - y_true\n    s = nasa_score(y_true, y_pred)\n    s_cap = nasa_score(np.minimum(y_true, cap), y_pred)\n    return {\"rmse\": float(np.sqrt(np.mean(d ** 2))),\n            \"rmse_capped\": float(np.sqrt(np.mean((y_pred - np.minimum(y_true, cap)) ** 2))),\n            \"mae\": float(np.mean(np.abs(d))), \"bias\": float(np.mean(d)),\n            \"late_pct\": float(np.mean(d > 0) * 100),\n            \"nasa_score\": float(s.sum()), \"nasa_mean\": float(s.mean()),\n            \"nasa_capped_mean\": float(s_cap.mean()), \"n\": int(len(d))}\n\n\ndef clf_metrics(y, p, threshold):\n    y, p = np.asarray(y), np.asarray(p)\n    pred = p >= threshold\n    tp, fp, fn = int((pred & (y == 1)).sum()), int((pred & (y == 0)).sum()), int((~pred & (y == 1)).sum())\n    prec = tp / (tp + fp) if tp + fp else 0.0\n    rec = tp / (tp + fn) if tp + fn else 0.0\n    two_classes = len(np.unique(y)) == 2\n    return {\"pr_auc\": float(average_precision_score(y, p)) if two_classes else float(\"nan\"),\n            \"roc_auc\": float(roc_auc_score(y, p)) if two_classes else float(\"nan\"),\n            \"precision\": prec, \"recall\": rec,\n            \"f2\": 5 * prec * rec / (4 * prec + rec) if prec + rec else 0.0,\n            \"positive_rate\": float(y.mean())}\n\n\ndef anomaly_metrics(P, cfg):\n    healthy = P.rul > cfg[\"anomaly_healthy_rul\"]\n    near = P.rul <= cfg[\"fail_window\"]\n    alarms = P[P.anomaly_persistent & ~healthy]\n    first = alarms.groupby(\"unit\").rul.max()\n    return {\"false_alarm_rate_healthy\": float(P.loc[healthy, \"anomaly_persistent\"].mean()) if healthy.any() else float(\"nan\"),\n            \"detection_rate_last30\": float(P.loc[near, \"anomaly_persistent\"].mean()) if near.any() else float(\"nan\"),\n            \"engines_detected_pct\": float(len(first) / P.unit.nunique() * 100),\n            \"median_rul_at_first_alarm\": float(first.median()) if len(first) else float(\"nan\")}\n\n\ndef interval_coverage(P):\n    # quantile models are trained on the capped target, so coverage is measured against it\n    return float(((P.rul_capped >= P.rul_lo) & (P.rul_capped <= P.rul_hi)).mean())\n\n\ndef coverage_by_band(P, name, bands=((0, 30), (30, 60), (60, 125))):\n    # coverage where decisions are made; CRITICAL relies on the lower bound when RUL <= 30\n    out = {}\n    for lo, hi in bands:\n        b = P[(P.rul > lo) & (P.rul <= hi)]\n        if len(b):\n            out[f\"{name}_coverage_rul_{lo}_{hi}\"] = interval_coverage(b)\n            out[f\"{name}_below_lower_bound_rul_{lo}_{hi}\"] = float((b.rul_capped < b.rul_lo).mean())\n    return out\n\n\ndef bootstrap_gain(P_old, P_new, cfg, n_boot=2000, seed=SEED):\n    # resample validation engines: how often is the new model better on capped RMSE, and how often is its\n    # capped NASA score more than the allowed ratio worse?\n    cap = cfg[\"rul_cap\"]\n    def per_engine(P):\n        P = test_like(P, cfg)\n        y = np.minimum(P.rul, cap)\n        return P.assign(se=(P.rul_pred - y) ** 2, nasa=nasa_score(y, P.rul_pred)).groupby(\"unit\")[[\"se\", \"nasa\"]] \\\n                .agg([\"sum\", \"count\"])\n    a = per_engine(P_old)\n    b = per_engine(P_new).reindex(a.index)\n    pick = np.random.default_rng(seed).integers(0, len(a), size=(n_boot, len(a)))\n    mean = lambda s, col: s[(col, \"sum\")].to_numpy()[pick].sum(axis=1) / s[(col, \"count\")].to_numpy()[pick].sum(axis=1)\n    diff = np.sqrt(mean(b, \"se\")) - np.sqrt(mean(a, \"se\"))\n    nasa_ratio = mean(b, \"nasa\") / np.maximum(mean(a, \"nasa\"), 1e-12)\n    return {\"prob_new_better\": float((diff < 0).mean()), \"prob_new_worse\": float((diff > 0).mean()),\n            \"prob_new_worse_by_margin\": float((diff > cfg[\"noninferiority_margin\"]).mean()),\n            \"rmse_diff_ci90\": [float(np.quantile(diff, 0.05)), float(np.quantile(diff, 0.95))],\n            \"prob_nasa_clearly_worse\": float((nasa_ratio > cfg[\"promote_max_nasa_ratio\"]).mean()),\n            \"nasa_ratio_ci90\": [float(np.quantile(nasa_ratio, 0.05)), float(np.quantile(nasa_ratio, 0.95))]}\n\n\ndef conformal_correction(y, lo, hi, coverage):\n    # conformalized quantile regression: how far outside [lo, hi] the truth falls, at the target coverage\n    scores = np.maximum(np.asarray(lo) - np.asarray(y), np.asarray(y) - np.asarray(hi))\n    level = min(1.0, np.ceil((len(scores) + 1) * coverage) / len(scores))\n    return float(np.quantile(scores, level))\n\n\ndef prefix(p, d):\n    return {f\"{p}_{k}\": v for k, v in d.items()}\n\n\ndef latest(P):\n    return P.sort_values([\"unit\", \"cycle\"]).groupby(\"unit\").tail(1).reset_index(drop=True)\n\n\ndef test_like(P, cfg):\n    return P[(P.cycle >= cfg[\"eval_min_cycles\"]) & (P.rul <= cfg[\"eval_max_rul\"])]\n\n\ndef train_bundle(df, kind, cfg, clf_threshold, interval_correction=0.0):\n    t0 = time.time()\n    prep = fit_prep(df, cfg)\n    F = featurize(df, prep, cfg)\n    feats, acols = feature_cols(prep, cfg), anomaly_cols(prep, cfg)\n    X, y = F[feats], F.rul_capped\n    first, last = cfg[\"anomaly_train_cycles\"]\n    healthy = F.cycle.between(first, last)  # early life: no failure labels used\n    iso = IsolationForest(n_estimators=300, random_state=SEED, n_jobs=-1).fit(F.loc[healthy, acols])\n    return {\n        \"kind\": kind, \"subset\": cfg[\"subset\"], \"cfg\": dict(cfg), \"prep\": prep,\n        \"features\": feats, \"anomaly_features\": acols,\n        \"rul_model\": make_rul_model(kind).fit(X, y),\n        \"q_lo\": make_quantile(0.1).fit(X, y), \"q_hi\": make_quantile(0.9).fit(X, y),\n        \"interval_correction\": float(interval_correction),\n        \"classifier\": make_classifier().fit(X, F.fail), \"clf_threshold\": float(clf_threshold),\n        \"iso\": iso,\n        \"anomaly_threshold\": float(np.quantile(-iso.score_samples(F.loc[healthy, acols]), cfg[\"anomaly_quantile\"])),\n        \"units\": sorted(int(u) for u in df.unit.unique()), \"n_rows\": int(len(df)),\n        \"train_seconds\": round(time.time() - t0, 1), \"created_at\": now_iso(),\n    }\n\n\ndef predict_bundle(b, df):\n    cfg, cap = b[\"cfg\"], b[\"cfg\"][\"rul_cap\"]\n    F = featurize(df, b[\"prep\"], cfg)\n    X = F[b[\"features\"]]\n    out = F[[c for c in META if c in F]].copy()\n    pred = np.clip(b[\"rul_model\"].predict(X), 0, cap)\n    q = b.get(\"interval_correction\", 0.0)\n    out[\"rul_pred\"] = pred\n    out[\"rul_lo\"] = np.minimum(np.clip(b[\"q_lo\"].predict(X) - q, 0, cap), pred)\n    out[\"rul_hi\"] = np.maximum(np.clip(b[\"q_hi\"].predict(X) + q, 0, cap), pred)\n    out[\"p_fail\"] = b[\"classifier\"].predict_proba(X)[:, 1]\n    out[\"fail_alert\"] = out.p_fail >= b[\"clf_threshold\"]\n    out[\"anomaly_score\"] = -b[\"iso\"].score_samples(F[b[\"anomaly_features\"]])\n    out[\"anomaly\"] = out.anomaly_score > b[\"anomaly_threshold\"]\n    k, w = cfg[\"anomaly_persist\"], cfg[\"roll_window\"]\n    g = out.groupby(\"unit\")\n    out[\"anomaly_persistent\"] = g.anomaly.transform(lambda a: a.astype(int).rolling(k, min_periods=k).min().fillna(0)).astype(bool)\n    out[\"recent_imputed\"] = g.imputed.transform(lambda a: a.astype(int).rolling(w, min_periods=1).max()).astype(bool)\n    out[\"recent_unseen\"] = g.unseen_condition.transform(lambda a: a.astype(int).rolling(w, min_periods=1).max()).astype(bool)\n    out[\"cycles_observed\"] = g.cumcount() + 1\n    return out\n\n\ndef evaluate_bundle(b, val_df, test_df):\n    # val_df=None skips validation metrics, for a model that has trained on the validation engines\n    cfg, cap = b[\"cfg\"], b[\"cfg\"][\"rul_cap\"]\n    Pt = predict_bundle(b, test_df)\n    last = latest(Pt)\n    m = prefix(\"test\", rul_metrics(last.rul, last.rul_pred, cap))\n    m[\"test_interval_coverage\"] = interval_coverage(last)\n    m.update(coverage_by_band(last, \"test\"))\n    Pv = None\n    if val_df is not None:\n        Pv = predict_bundle(b, val_df)\n        tl = test_like(Pv, cfg)\n        m.update(prefix(\"val\", rul_metrics(tl.rul, tl.rul_pred, cap)))\n        m.update(prefix(\"val_all\", rul_metrics(Pv.rul, Pv.rul_pred, cap)))\n        m[\"val_interval_coverage\"] = interval_coverage(tl)\n        m.update(coverage_by_band(tl, \"val\"))\n        m.update(prefix(\"val_clf\", clf_metrics(Pv.fail, Pv.p_fail, b[\"clf_threshold\"])))\n        m.update(prefix(\"val_anom\", anomaly_metrics(Pv, cfg)))\n    return m, Pv, Pt\n\n\ndef show_metrics(m, keys=(\"val_rmse_capped\", \"val_rmse\", \"val_nasa_capped_mean\", \"test_rmse_capped\", \"test_rmse\",\n                          \"test_nasa_score\", \"test_mae\", \"test_bias\",\n                          \"test_late_pct\", \"val_interval_coverage\", \"test_interval_coverage\", \"val_clf_pr_auc\", \"val_clf_recall\",\n                          \"val_clf_precision\", \"val_anom_false_alarm_rate_healthy\", \"val_anom_median_rul_at_first_alarm\")):\n    return pd.Series({k: round(m[k], 3) for k in keys if k in m})",
"# ---- deep sequence models (PyTorch): 1D-CNN, GRU and TCN on windows of past normalised sensor readings ----\nimport os\nos.environ.setdefault(\"CUBLAS_WORKSPACE_CONFIG\", \":4096:8\")  # deterministic cuBLAS (set before CUDA starts)\nimport torch\nfrom torch import nn\n\nDEVICE = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\ntorch.backends.cudnn.benchmark = False\ntorch.backends.cudnn.deterministic = True\ntorch.use_deterministic_algorithms(True, warn_only=True)\nprint(\"torch\", torch.__version__, \"| device\", DEVICE)\n\n\nclass CNN1D(nn.Module):\n    # the classic C-MAPSS conv net: stacked 1D convolutions over the window, then mean + last-step pooling\n    def __init__(self, c_in, width=64, dropout=0.2):\n        super().__init__()\n        self.conv = nn.Sequential(nn.Conv1d(c_in, 32, 5, padding=2), nn.ReLU(),\n                                  nn.Conv1d(32, width, 5, padding=2), nn.ReLU(),\n                                  nn.Conv1d(width, width, 3, padding=1), nn.ReLU())\n        self.head = nn.Sequential(nn.Linear(2 * width, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))\n\n    def forward(self, x):  # x: (batch, time, channels)\n        h = self.conv(x.transpose(1, 2))\n        return self.head(torch.cat([h.mean(-1), h[..., -1]], 1)).squeeze(-1)\n\n\nclass GRUNet(nn.Module):\n    # recurrent: two stacked GRU layers, read out at the latest cycle\n    def __init__(self, c_in, hidden=64, layers=2, dropout=0.2):\n        super().__init__()\n        self.gru = nn.GRU(c_in, hidden, layers, batch_first=True, dropout=dropout)\n        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))\n\n    def forward(self, x):\n        out, _ = self.gru(x)\n        return self.head(out[:, -1]).squeeze(-1)\n\n\nclass TCNBlock(nn.Module):\n    # causal dilated convolutions (left padding only) with a residual connection\n    def __init__(self, c_in, c_out, dilation, k=3, dropout=0.1):\n        super().__init__()\n        self.pad = (k - 1) * dilation\n        self.c1 = nn.Conv1d(c_in, c_out, k, dilation=dilation)\n        self.c2 = nn.Conv1d(c_out, c_out, k, dilation=dilation)\n        self.drop = nn.Dropout(dropout)\n        self.skip = nn.Conv1d(c_in, c_out, 1) if c_in != c_out else nn.Identity()\n\n    def forward(self, x):\n        h = self.drop(torch.relu(self.c1(nn.functional.pad(x, (self.pad, 0)))))\n        h = self.drop(torch.relu(self.c2(nn.functional.pad(h, (self.pad, 0)))))\n        return torch.relu(h + self.skip(x))\n\n\nclass TCN(nn.Module):\n    def __init__(self, c_in, width=64, dilations=(1, 2, 4, 8)):\n        super().__init__()\n        blocks, c = [], c_in\n        for d in dilations:  # receptive field 61 cycles >= the 30-cycle window\n            blocks.append(TCNBlock(c, width, d))\n            c = width\n        self.net = nn.Sequential(*blocks)\n        self.head = nn.Linear(width, 1)\n\n    def forward(self, x):\n        return self.head(self.net(x.transpose(1, 2))[..., -1]).squeeze(-1)\n\n\nNETS = {\"cnn\": CNN1D, \"gru\": GRUNet, \"tcn\": TCN}\n\n\ndef _batches(n, size, rng=None):\n    idx = rng.permutation(n) if rng is not None else np.arange(n)\n    for i in range(0, n, size):\n        yield idx[i:i + size]\n\n\ndef predict_dl(model, X, cap, batch=4096):\n    model.eval()\n    out = []\n    with torch.no_grad():\n        for b in _batches(len(X), batch):\n            out.append(model(torch.from_numpy(X[b]).to(DEVICE)).float().cpu().numpy())\n    return np.clip(np.concatenate(out) * cap, 0, cap) if out else np.zeros(0)\n\n\ndef fit_dl(kind, Xtr, ytr, Xes, yes, dl, seed=SEED):\n    # MSE on the capped target scaled to [0, 1]; early stopping on engines the net does not train on.\n    # `seed` sets the initial weights and the batch order, the two sources of run-to-run spread in a net\n    torch.manual_seed(seed)\n    rng = np.random.default_rng(seed)\n    model = NETS[kind](Xtr.shape[2]).to(DEVICE)\n    opt = torch.optim.AdamW(model.parameters(), lr=dl[\"lr\"], weight_decay=dl[\"weight_decay\"])\n    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)\n    # the training windows fit in GPU memory (largest subset ~0.1 GB), so batches are gathered on the device\n    Xt = torch.from_numpy(Xtr).to(DEVICE)\n    yt = torch.from_numpy((ytr / dl[\"cap\"]).astype(np.float32)).to(DEVICE)\n    best, best_epoch, bad, t0 = np.inf, 0, 0, time.time()\n    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}\n    for epoch in range(dl[\"epochs\"]):\n        model.train()\n        for b in _batches(len(Xt), dl[\"batch\"], rng):\n            b = torch.from_numpy(b).to(DEVICE)\n            opt.zero_grad()\n            loss = nn.functional.mse_loss(model(Xt[b]), yt[b])\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            opt.step()\n        rmse = float(np.sqrt(np.mean((predict_dl(model, Xes, dl[\"cap\"]) - yes) ** 2)))\n        sched.step(rmse)\n        if rmse < best - 1e-3:\n            best, best_epoch, bad = rmse, epoch + 1, 0\n            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        else:\n            bad += 1\n            if bad >= dl[\"patience\"]:\n                break\n    model.load_state_dict(best_state)\n    return model, {\"best_epoch\": best_epoch, \"epochs_run\": epoch + 1, \"early_stop_rmse_capped\": round(best, 3),\n                   \"seconds\": round(time.time() - t0, 1), \"params\": sum(p.numel() for p in model.parameters())}",
"# ---- deep models vs gradient boosting: same engines, same 5 CV folds, same truth, then the official test set ----\nDL = {\"window\": 30, \"epochs\": 60, \"patience\": 8, \"batch\": 512, \"lr\": 1e-3, \"weight_decay\": 1e-4,\n      \"es_frac\": 0.15, \"cap\": 125,\n      # each net is trained from 3 random starts; the deployable net is their average (a \"seed ensemble\"),\n      # and the per-seed scores show how much of a difference is luck of the initialisation\n      \"seeds\": [42, 1, 2], **globals().get(\"DL_OVERRIDES\", {})}\nEVAL_CAP = 125  # every model is scored against min(true RUL, 125)\nDEEP = list(NETS)\nTREES = [\"hgb\"] + ([\"xgb\"] if xgb is not None else [])\ndev = train_df[train_df.unit.isin(SPLITS[\"initial\"] + SPLITS[\"new\"])].reset_index(drop=True)  # = v2's training set\nfolds = list(GroupKFold(n_splits=CFG[\"cv_folds\"]).split(dev, groups=dev.unit))  # identical to the ablation folds\n\n\ndef windows(dz, sensors, w):\n    # one window per row: the last w cycles up to and including that row (past only), left-padded with the\n    # engine's first reading; channels = the normalised sensors + cycle / 100 (the trees get cycle too)\n    blocks = []\n    for _, g in dz.groupby(\"unit\", sort=True):\n        y = np.column_stack([g[sensors].to_numpy(np.float32), g.cycle.to_numpy(np.float32) / 100.0])\n        y = np.vstack([np.repeat(y[:1], w - 1, axis=0), y])\n        blocks.append(np.lib.stride_tricks.sliding_window_view(y, w, axis=0).transpose(0, 2, 1))\n    return np.ascontiguousarray(np.concatenate(blocks), dtype=np.float32)\n\n\ndef check_window_causality(df, prep, w, n_units=4):\n    # a row's window must not change when the engine's future cycles are removed\n    full = windows(apply_prep(df, prep), prep[\"sensors\"], w)\n    z = apply_prep(df, prep)\n    for u in z.unit.unique()[:n_units]:\n        g = df[df.unit == u]\n        cut = int(g.cycle.max() * 0.6)\n        part = windows(apply_prep(g[g.cycle <= cut], prep), prep[\"sensors\"], w)\n        row = np.flatnonzero((z.unit == u).to_numpy() & (z.cycle == cut).to_numpy())[0]\n        assert np.allclose(part[-1], full[row]), f\"window leak on engine {u}\"\n\n\ndef test_like_mask(F):\n    return ((F.cycle >= CFG[\"eval_min_cycles\"]) & (F.rul <= CFG[\"eval_max_rul\"])).to_numpy()\n\n\ndef fit_all(tr, tr_dl, es, targets):\n    # trees train on `tr`; nets train on `tr_dl` and early-stop on `es`; one preprocessing fit on `tr` for both.\n    # Returns predictions on every target frame (rows in apply_prep order) and the nets' training info.\n    prep = fit_prep(tr, CFG)\n    cols, sensors, w = feature_cols(prep, CFG), prep[\"sensors\"], DL[\"window\"]\n    Ftr = featurize(tr, prep, CFG)\n    F = {k: featurize(d, prep, CFG) for k, d in targets.items()}\n    preds = {k: {} for k in targets}\n    for kind in TREES:\n        m = make_rul_model(kind).fit(Ftr[cols], np.minimum(Ftr.rul, DL[\"cap\"]))\n        for k in targets:\n            preds[k][kind] = np.clip(m.predict(F[k][cols]), 0, DL[\"cap\"])\n    Ztr, Zes = apply_prep(tr_dl, prep), apply_prep(es, prep)\n    Xtr, Xes = windows(Ztr, sensors, w), windows(Zes, sensors, w)\n    es_rows = ((Zes.cycle >= CFG[\"eval_min_cycles\"]) & (Zes.rul <= CFG[\"eval_max_rul\"])).to_numpy()\n    if not es_rows.any():\n        es_rows[:] = True\n    Xes, yes = Xes[es_rows], np.minimum(Zes.rul.to_numpy()[es_rows], EVAL_CAP).astype(float)\n    Xt = {k: windows(apply_prep(d, prep), sensors, w) for k, d in targets.items()}\n    info, ytr = {}, np.minimum(Ztr.rul.to_numpy(), DL[\"cap\"]).astype(np.float32)\n    for kind in DEEP:\n        for s in DL[\"seeds\"]:\n            model, info[f\"{kind} [seed {s}]\"] = fit_dl(kind, Xtr, ytr, Xes, yes, DL, seed=s)\n            for k in targets:\n                preds[k][f\"{kind} [seed {s}]\"] = predict_dl(model, Xt[k], DL[\"cap\"])\n            del model\n        for k in targets:  # the net as deployed: average of its seeds\n            preds[k][kind] = np.mean([preds[k][f\"{kind} [seed {s}]\"] for s in DL[\"seeds\"]], axis=0)\n    for k in targets:  # ensembles: trees + each net, and all three nets together\n        p = preds[k]\n        for kind in DEEP:\n            p[f\"hgb + {kind}\"] = (p[\"hgb\"] + p[kind]) / 2\n        p[\"cnn + gru + tcn\"] = np.mean([p[kind] for kind in DEEP], axis=0)\n        p[\"hgb + cnn + gru + tcn\"] = (p[\"hgb\"] + p[\"cnn + gru + tcn\"]) / 2\n    return preds, F, info, prep\n\n\ndef scores(y, p):\n    near = y <= 60\n    return {\"rmse_capped\": float(np.sqrt(np.mean((p - np.minimum(y, EVAL_CAP)) ** 2))),\n            \"rmse_true\": float(np.sqrt(np.mean((p - y) ** 2))),\n            \"rmse_true_rul_le_60\": float(np.sqrt(np.mean((p[near] - y[near]) ** 2))) if near.any() else float(\"nan\"),\n            \"nasa_capped_mean\": float(nasa_score(np.minimum(y, EVAL_CAP), p).mean())}\n\n\ncheck_window_causality(dev, fit_prep(dev, CFG), DL[\"window\"])\nprint(f\"[{SUBSET}] window causality check passed; {dev.unit.nunique()} engines, {len(folds)} folds, window {DL['window']}\")\n\n# 1) cross-validation on the training engines (paired folds; nets early-stop on 15% of each fold's training engines)\noof, fit_info, t0 = [], [], time.time()\nfor k, (tr_idx, va_idx) in enumerate(folds):\n    tr, va = dev.iloc[tr_idx], dev.iloc[va_idx]\n    es_units = sorted(tr.unit.unique())\n    np.random.default_rng(SEED + k).shuffle(es_units)\n    es_units = set(es_units[:max(1, round(len(es_units) * DL[\"es_frac\"]))])\n    preds, F, info, _ = fit_all(tr, tr[~tr.unit.isin(es_units)], tr[tr.unit.isin(es_units)], {\"va\": va})\n    rows = test_like_mask(F[\"va\"])\n    y = F[\"va\"].rul.to_numpy()[rows]\n    for name, p in preds[\"va\"].items():\n        oof.append({\"fold\": k, \"model\": name, **scores(y, p[rows])})\n    fit_info += [{\"stage\": f\"cv fold {k}\", \"model\": m, **v} for m, v in info.items()]\n    print(f\"[{SUBSET}] fold {k}: \" + \", \".join(f\"{m} {s['rmse_capped']:.2f}\" for m, s in\n          ((r[\"model\"], r) for r in oof if r[\"fold\"] == k and r[\"model\"] in TREES + DEEP)) + f\" ({time.time() - t0:.0f}s)\")\n\ncv = pd.DataFrame(oof)\nbase = cv[cv.model == \"hgb\"].set_index(\"fold\").rmse_capped\nrows_ = []\nfor name, g in cv.groupby(\"model\", sort=False):\n    delta = g.set_index(\"fold\").rmse_capped - base\n    rows_.append({\"model\": name, **g.drop(columns=[\"fold\", \"model\"]).mean().to_dict(),\n                  \"fold_std_rmse_capped\": float(g.rmse_capped.std()), \"delta_vs_hgb\": float(delta.mean()),\n                  \"folds_better_than_hgb\": f\"{int((delta < 0).sum())}/{len(delta)}\"})\ncv_table = pd.DataFrame(rows_).set_index(\"model\").round(3)\nPER_SEED = [m for m in cv_table.index if \"[seed \" in m]\n\n# 2) official test set: trees on the same engines as the production model (v2); nets early-stop on the\n#    validation engines (never used for training here), one prediction per test engine at its last cycle\npreds, F, info, prep = fit_all(dev, dev, val_df, {\"test\": test_df})\nfit_info += [{\"stage\": \"test fit\", \"model\": m, **v} for m, v in info.items()]\nlast = F[\"test\"].groupby(\"unit\").cycle.idxmax().to_numpy()\ny_test = F[\"test\"].rul.to_numpy()[last]\ntest_rows = []\nfor name, p in preds[\"test\"].items():\n    pt = p[last]\n    test_rows.append({\"model\": name, \"test_rmse_capped\": float(np.sqrt(np.mean((pt - np.minimum(y_test, EVAL_CAP)) ** 2))),\n                      \"test_rmse\": float(np.sqrt(np.mean((pt - y_test) ** 2))),\n                      \"test_nasa_score\": float(nasa_score(y_test, pt).sum()),\n                      \"test_nasa_capped_mean\": float(nasa_score(np.minimum(y_test, EVAL_CAP), pt).mean())})\ntest_table = pd.DataFrame(test_rows).set_index(\"model\").round(3)\n\n# how much of each net's score is luck of the random start: spread over its seeds, vs its seed ensemble\nseed_rows = []\nfor kind in DEEP:\n    single = [f\"{kind} [seed {s}]\" for s in DL[\"seeds\"]]\n    cvs, tests = cv_table.loc[single, \"rmse_capped\"], test_table.loc[single, \"test_rmse_capped\"]\n    seed_rows.append({\"net\": kind, \"cv_single_seed_mean\": cvs.mean(), \"cv_single_seed_std\": cvs.std(),\n                      \"cv_single_seed_range\": cvs.max() - cvs.min(), \"cv_seed_ensemble\": cv_table.loc[kind, \"rmse_capped\"],\n                      \"cv_single_seed_delta_vs_hgb_mean\": cv_table.loc[single, \"delta_vs_hgb\"].mean(),\n                      \"test_single_seed_mean\": tests.mean(), \"test_single_seed_range\": tests.max() - tests.min(),\n                      \"test_seed_ensemble\": test_table.loc[kind, \"test_rmse_capped\"]})\nseed_table = pd.DataFrame(seed_rows).set_index(\"net\").round(3)\n\n# 3) verdict, same spirit as the promotion gate: a net replaces boosting only if it is clearly better\n#    (capped CV RMSE lower by more than the 0.5-cycle margin, in at least 4 of 5 paired folds). Candidates are\n#    the deployable models: the seed-ensembled nets and the blends (single-seed runs are diagnostics only)\ncand = cv_table.drop(index=TREES + PER_SEED)\nbest = cand.rmse_capped.idxmin()\nwins = int(cand.loc[best, \"folds_better_than_hgb\"].split(\"/\")[0])\nadopt = cand.loc[best, \"delta_vs_hgb\"] < -CFG[\"noninferiority_margin\"] and wins >= 4\nverdict = {\"best_non_tree_model\": best, \"delta_vs_hgb_cycles\": float(cand.loc[best, \"delta_vs_hgb\"]),\n           \"folds_better\": cand.loc[best, \"folds_better_than_hgb\"], \"adopt_as_challenger\": bool(adopt),\n           \"decision\": (f\"{best} enters the promotion gate as a challenger\" if adopt\n                        else \"keep gradient boosting: no deep model or blend is clearly better\")}\n\ndisplay(cv_table.drop(index=PER_SEED))\ndisplay(test_table.drop(index=PER_SEED))\ndisplay(seed_table)\nprint(verdict)\nDEEP_OUT = OUT / \"deep\"\nDEEP_OUT.mkdir(parents=True, exist_ok=True)\ncv_table.to_csv(DEEP_OUT / \"cv_table.csv\")\ncv.to_csv(DEEP_OUT / \"cv_per_fold.csv\", index=False)\ntest_table.to_csv(DEEP_OUT / \"test_table.csv\")\nseed_table.to_csv(DEEP_OUT / \"seed_spread.csv\")\npd.DataFrame(fit_info).to_csv(DEEP_OUT / \"training_info.csv\", index=False)\nsave_json({\"subset\": SUBSET, \"engines\": int(dev.unit.nunique()), \"folds\": CFG[\"cv_folds\"], \"dl_config\": DL,\n           \"sensors\": prep[\"sensors\"], \"device\": str(DEVICE), \"verdict\": verdict,\n           \"cv\": cv_table.reset_index().to_dict(orient=\"records\"),\n           \"test\": test_table.reset_index().to_dict(orient=\"records\"),\n           \"seed_spread\": seed_table.reset_index().to_dict(orient=\"records\")}, DEEP_OUT / \"deep.json\")\nDEEP_RESULT = {\"cv\": cv_table, \"test\": test_table, \"seeds\": seed_table, \"verdict\": verdict,\n               \"info\": pd.DataFrame(fit_info)}"
]''')
print(len(DEEP_CELLS), 'cells loaded')

In [ ]:
import gc, shutil, time, traceback
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
WORK = Path("/kaggle/working") if Path("/kaggle/input").exists() else Path.cwd() / "outputs"
shutil.rmtree(WORK / "artifacts", ignore_errors=True)

deep = {}
for subset in SUBSETS:
    print(f"\n{'=' * 34} {subset} {'=' * 34}")
    ns = {"__name__": "__main__", "display": display}
    t0 = time.time()
    try:
        for i, src in enumerate(DEEP_CELLS):
            exec(compile(src.replace('SUBSET = "FD001"', f'SUBSET = "{subset}"'), f"<{subset} cell {i}>", "exec"), ns)
        deep[subset] = ns["DEEP_RESULT"]
    except Exception:
        traceback.print_exc()
    finally:
        plt.close("all")
        if "torch" in ns:
            ns["torch"].cuda.empty_cache()
        del ns
        gc.collect()
    print(f"{subset}: {'ok' if subset in deep else 'FAILED'} in {(time.time() - t0) / 60:.1f} min")

## Cross-subset summary

`delta_vs_hgb` is the change in capped CV RMSE (cycles) against HistGradientBoosting, averaged over the 5 paired folds: negative = better than boosting. The test table is the official NASA test set (capped and true RMSE, NASA score).

In [ ]:
if deep:
    order = list(dict.fromkeys(m for r in deep.values() for m in r["cv"].index if "[seed " not in m))
    cv_rmse = pd.DataFrame({s: r["cv"]["rmse_capped"] for s, r in deep.items()}).reindex(order)
    cv_delta = pd.DataFrame({s: r["cv"]["delta_vs_hgb"] for s, r in deep.items()}).reindex(order)
    cv_wins = pd.DataFrame({s: r["cv"]["folds_better_than_hgb"] for s, r in deep.items()}).reindex(order)
    test_capped = pd.DataFrame({s: r["test"]["test_rmse_capped"] for s, r in deep.items()}).reindex(order)
    test_true = pd.DataFrame({s: r["test"]["test_rmse"] for s, r in deep.items()}).reindex(order)
    verdicts = pd.DataFrame({s: r["verdict"] for s, r in deep.items()})
    seeds = pd.concat({s: r["seeds"] for s, r in deep.items()})
    for title, t in [("capped RMSE (cycles), CV on training engines; nets = average of 3 seeds", cv_rmse),
                     ("change vs HistGB (cycles; - = better)", cv_delta.round(2)),
                     ("folds where the model beat HistGB", cv_wins),
                     ("official test set, capped RMSE", test_capped), ("official test set, true RMSE", test_true),
                     ("seed spread per net (single seeds vs their average)", seeds), ("verdict", verdicts)]:
        print(title)
        display(t)
    out = WORK / "artifacts"
    out.mkdir(parents=True, exist_ok=True)
    cv_rmse.to_csv(out / "deep_cv_rmse_capped.csv")
    cv_delta.to_csv(out / "deep_cv_delta_vs_hgb.csv")
    cv_wins.to_csv(out / "deep_cv_folds_better.csv")
    test_capped.to_csv(out / "deep_test_rmse_capped.csv")
    test_true.to_csv(out / "deep_test_rmse.csv")
    verdicts.to_csv(out / "deep_verdicts.csv")
    seeds.to_csv(out / "deep_seed_spread.csv")
    pd.concat({s: r["info"] for s, r in deep.items()}).to_csv(out / "deep_training_info.csv")
    zip_path = shutil.make_archive(str(WORK / "deep_outputs"), "zip", root_dir=out)
    print("outputs:", zip_path)